# Incremental Learning Strategies — OE WC30

Compares five incremental learning approaches on OE WC30.

**Data split:**
- Initial training: first 70% of days
- Update batches 1-3: next 15% of days split into 3 equal batches
- Test set: last 15% of days — fixed, never touched

**Strategies compared:**
1. Baseline — train on initial data only, no updates
2. Naive update — add new trees with no weighting
3. Option 1a — Exponential time decay weighting
4. Option 1b — Linear time decay weighting
5. Option 2 — Rolling window (drop old data)
6. Option 3a — KS-test drift detection
7. Option 3b — ADWIN drift detection

**Metric:** chunk-level MAE/task on the fixed test set after each batch.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy import stats
from sklearn.metrics import mean_absolute_error, r2_score

import importlib
import feature_engineer
importlib.reload(feature_engineer)
from feature_engineer import get_engineered_df

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

PATH         = Path('../data/processed')
WAREHOUSE    = 'OE'
WORKCODE     = '30'
MAX_TIME     = 300
BLOCK_SIZE   = 50
RANDOM_STATE = 2026
UPDATE_TREES = 100   # trees added per batch in each strategy

NOT_AVAILABLE = [
    'Travel_Distance',
    'same_aisle', 'same_lockey', 'same_location', 'same_level', 'diff_level',
    'time_of_day', 'day_of_week', 'hour',
]

XGB_PARAMS = dict(
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective='reg:tweedie',
    tweedie_variance_power=1.3,
    tree_method='hist',
    seed=RANDOM_STATE,
)


In [ ]:
def resolve_data_path(warehouse):
    return PATH / f'{warehouse.lower()}_detailed.parquet'


def load_engineered_data(warehouse, workcode, max_time=MAX_TIME):
    d, features_all, cat_cols_all = get_engineered_df(
        file_path=resolve_data_path(warehouse),
        warehouse=warehouse,
        max_time=max_time,
        work_code=workcode,
    )
    d = d.copy()
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], errors='coerce')
    d = d.dropna(subset=['Timestamp']).copy()
    d['date']     = d['Timestamp'].dt.date
    d['WorkCode'] = d['WorkCode'].astype(str).str.replace('.0', '', regex=False)
    features = [f for f in features_all if f not in NOT_AVAILABLE]
    cat_cols = [c for c in cat_cols_all if c not in NOT_AVAILABLE]
    return d, features, cat_cols


def make_X_single(df, features, cat_cols, train_columns=None):
    """One-hot encode a single dataframe and align to train_columns if given."""
    X = pd.get_dummies(df[features], columns=cat_cols, drop_first=True)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    if train_columns is not None:
        X = X.reindex(columns=train_columns, fill_value=0)
    return X


def make_test_blocks(test_df, block_size=BLOCK_SIZE):
    """Copied from final_oe.ipynb — groups test picks into fixed-size chunks."""
    d = test_df.sort_values(['UserID', 'Timestamp']).copy()
    blocks, block_rows = [], []
    for (uid, day), g in d.groupby(['UserID', 'date'], sort=False):
        g = g.sort_values('Timestamp').reset_index().rename(columns={'index': 'orig_index'}).copy()
        for start in range(0, len(g), block_size):
            chunk = g.iloc[start:start + block_size].copy()
            if len(chunk) < block_size:
                continue
            if chunk['WorkCode'].nunique() != 1:
                continue
            if (chunk['Time_Delta_sec'] > MAX_TIME).any():
                continue
            block_id = f'{uid}_{day}_{start // block_size}'
            chunk['BlockID'] = block_id
            block_rows.append(chunk)
            blocks.append({
                'BlockID':     block_id,
                'UserID':      uid,
                'date':        day,
                'WorkCode':    chunk['WorkCode'].iloc[0],
                'n_tasks':     len(chunk),
                'actual_time': chunk['Time_Delta_sec'].sum(),
            })
    block_df      = pd.DataFrame(blocks)
    block_rows_df = pd.concat(block_rows, ignore_index=True) if block_rows else pd.DataFrame()
    return block_df, block_rows_df


def eval_blocks(test_df, preds_array, train_columns):
    """
    Given test_df and pick-level predictions, build blocks and return
    chunk-level MAE/task and R².
    """
    test_df = test_df.copy().reset_index(drop=True)
    block_df, block_rows_df = make_test_blocks(test_df)
    if block_df.empty:
        return np.nan, np.nan, 0

    temp = test_df.copy().reset_index().rename(columns={'index': 'orig_index'})
    temp['pred'] = preds_array
    block_rows_df = block_rows_df.merge(temp[['orig_index', 'pred']], on='orig_index', how='left')

    block_pred = (
        block_rows_df.groupby('BlockID')
        .agg(actual_time=('Time_Delta_sec', 'sum'), pred=('pred', 'sum'))
        .reset_index()
    )
    mae  = mean_absolute_error(block_pred['actual_time'], block_pred['pred'])
    r2   = r2_score(block_pred['actual_time'], block_pred['pred'])
    return mae / BLOCK_SIZE, r2, len(block_pred)


In [ ]:
# ── Load and split data (row-based — only 8 days available) ──────────────
# With only 8 days, splitting by day gives too few rows per batch.
# Instead we sort chronologically by timestamp and split by row count.
# This simulates sequential batches of new picks arriving over time.
df, features, cat_cols = load_engineered_data(WAREHOUSE, WORKCODE)

# Sort chronologically
df = df.sort_values(['date', 'Timestamp']).reset_index(drop=True)
n_rows = len(df)

# Chronological split: 70% train | 15% update (3 batches of 5%) | 15% test
n_train  = int(n_rows * 0.70)
n_test   = int(n_rows * 0.15)
n_update = n_rows - n_train - n_test
n_batch  = n_update // 3

train_df = df.iloc[:n_train].copy().reset_index(drop=True)
batch_dfs = [
    df.iloc[n_train          : n_train + n_batch].copy().reset_index(drop=True),
    df.iloc[n_train + n_batch: n_train + n_batch*2].copy().reset_index(drop=True),
    df.iloc[n_train + n_batch*2: n_train + n_update].copy().reset_index(drop=True),
]
test_df = df.iloc[n_train + n_update:].copy().reset_index(drop=True)

print(f'Total rows: {n_rows:,}')
print(f'Train rows: {len(train_df):,} (rows 0–{n_train-1})')
for i, bdf in enumerate(batch_dfs):
    print(f'Batch {i+1} rows: {len(bdf):,} | dates: {bdf["date"].min()} to {bdf["date"].max()}')
print(f'Test rows:  {len(test_df):,} (fixed held-out set)')
print(f'Test dates: {test_df["date"].min()} to {test_df["date"].max()}')

# Build fixed test feature matrix — never changes across experiments
X_train_init  = make_X_single(train_df, features, cat_cols)
train_columns = X_train_init.columns.tolist()  # column order locked from initial training
X_test        = make_X_single(test_df, features, cat_cols, train_columns=train_columns)
y_test        = test_df['Time_Delta_sec'].astype(float)

# Dict to collect results: strategy -> list of MAE/task after each batch
results = {}
print('\nData split ready.')


In [ ]:
# ── Strategy 0: Baseline — train on initial data only, no updates ────────
print('Training baseline model...')

y_train = train_df['Time_Delta_sec'].astype(float)
dtrain  = xgb.DMatrix(X_train_init, label=y_train)
dtest   = xgb.DMatrix(X_test)

baseline_model = xgb.train(
    XGB_PARAMS, dtrain,
    num_boost_round=1200,
    verbose_eval=False,
)

preds = baseline_model.predict(dtest)
mae_per_task, r2, n_blocks = eval_blocks(test_df, preds, train_columns)
print(f'Baseline | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f} | Blocks: {n_blocks}')

# Baseline is a flat line — same result at every 'batch' since model never updates
results['Baseline'] = [mae_per_task, mae_per_task, mae_per_task, mae_per_task]
print('Done.')


In [ ]:
# ── Strategy 1: Naive update — add trees with no weighting ──────────────
# All new data treated equally regardless of age.
# This is what our current update_model.py does.
print('Running naive incremental update...')

model_naive = xgb.Booster()
model_naive.load_model(dict(baseline_model.save_raw()))  # start from baseline

# Re-train baseline fresh to get a booster object
model_naive = xgb.train(XGB_PARAMS, dtrain, num_boost_round=1200, verbose_eval=False)

naive_maes = []
preds = model_naive.predict(dtest)
mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
naive_maes.append(mae_per_task)
print(f'After initial training | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f}')

for i, batch_df in enumerate(batch_dfs):
    X_batch = make_X_single(batch_df, features, cat_cols, train_columns=train_columns)
    y_batch = batch_df['Time_Delta_sec'].astype(float)
    d_batch = xgb.DMatrix(X_batch, label=y_batch)

    model_naive = xgb.train(
        XGB_PARAMS, d_batch,
        num_boost_round=UPDATE_TREES,
        xgb_model=model_naive,
        verbose_eval=False,
    )
    preds = model_naive.predict(dtest)
    mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
    naive_maes.append(mae_per_task)
    print(f'After batch {i+1} | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f} | Trees: {model_naive.num_boosted_rounds()}')

results['Naive update'] = naive_maes
print('Done.')


In [ ]:
# ── Option 1a: Exponential time decay weighting ──────────────────────────
# weight = exp(-lambda * days_old)
# Recent picks get weight ~1.0, older picks decay toward 0.
# Lambda controls how fast weights decay:
#   lambda=0.01 → half-life ~70 days (slow decay)
#   lambda=0.05 → half-life ~14 days (fast decay)
print('Running exponential decay strategy...')

LAMBDA = 0.02  # tune this — try 0.01 (slow) and 0.05 (fast)

def exponential_weights(df, reference_date, lam=LAMBDA):
    """Compute per-row weights based on days from reference_date."""
    days_old = (pd.Timestamp(reference_date) - pd.to_datetime(df['date'])).dt.days
    days_old = days_old.clip(lower=0)
    return np.exp(-lam * days_old).values


model_exp = xgb.train(XGB_PARAMS, dtrain, num_boost_round=1200, verbose_eval=False)

exp_maes = []
preds = model_exp.predict(dtest)
mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
exp_maes.append(mae_per_task)
print(f'After initial training | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f}')

# Accumulate all data seen so far for weighted retraining
seen_df = train_df.copy()

for i, batch_df in enumerate(batch_dfs):
    seen_df = pd.concat([seen_df, batch_df], ignore_index=True)
    reference_date = seen_df['date'].max()  # most recent date seen

    X_seen = make_X_single(seen_df, features, cat_cols, train_columns=train_columns)
    y_seen = seen_df['Time_Delta_sec'].astype(float)
    w_seen = exponential_weights(seen_df, reference_date)

    # Pass weights to DMatrix — XGBoost applies them during tree building
    d_seen = xgb.DMatrix(X_seen, label=y_seen, weight=w_seen)

    model_exp = xgb.train(
        XGB_PARAMS, d_seen,
        num_boost_round=UPDATE_TREES,
        xgb_model=model_exp,
        verbose_eval=False,
    )
    preds = model_exp.predict(dtest)
    mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
    exp_maes.append(mae_per_task)
    w_mean = w_seen.mean()
    print(f'After batch {i+1} | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f} | Mean weight: {w_mean:.3f}')

results['Exp decay (λ=0.02)'] = exp_maes
print('Done.')


In [ ]:
# ── Option 1b: Linear time decay weighting ───────────────────────────────
# weight = max(0, 1 - days_old / max_days)
# Simpler than exponential — weight drops linearly to 0 at max_days.
# Easier to explain: 'data older than 90 days gets weight 0'
print('Running linear decay strategy...')

MAX_DECAY_DAYS = 90  # data older than this gets weight 0

def linear_weights(df, reference_date, max_days=MAX_DECAY_DAYS):
    """Compute per-row weights that decay linearly to 0 at max_days."""
    days_old = (pd.Timestamp(reference_date) - pd.to_datetime(df['date'])).dt.days
    days_old = days_old.clip(lower=0)
    return np.clip(1 - days_old / max_days, 0, 1).values


model_lin = xgb.train(XGB_PARAMS, dtrain, num_boost_round=1200, verbose_eval=False)

lin_maes = []
preds = model_lin.predict(dtest)
mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
lin_maes.append(mae_per_task)
print(f'After initial training | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f}')

seen_df = train_df.copy()

for i, batch_df in enumerate(batch_dfs):
    seen_df = pd.concat([seen_df, batch_df], ignore_index=True)
    reference_date = seen_df['date'].max()

    X_seen = make_X_single(seen_df, features, cat_cols, train_columns=train_columns)
    y_seen = seen_df['Time_Delta_sec'].astype(float)
    w_seen = linear_weights(seen_df, reference_date)

    d_seen = xgb.DMatrix(X_seen, label=y_seen, weight=w_seen)
    model_lin = xgb.train(
        XGB_PARAMS, d_seen,
        num_boost_round=UPDATE_TREES,
        xgb_model=model_lin,
        verbose_eval=False,
    )
    preds = model_lin.predict(dtest)
    mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
    lin_maes.append(mae_per_task)
    n_zero = (w_seen == 0).sum()
    print(f'After batch {i+1} | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f} | Zero-weight rows: {n_zero:,}')

results['Linear decay (90d)'] = lin_maes
print('Done.')


In [ ]:
# ── Option 2: Rolling window — only keep last N days ─────────────────────
# Simplest approach. Drop anything older than WINDOW_DAYS.
# Brittle if warehouse changes slowly over more than WINDOW_DAYS.
print('Running rolling window strategy...')

WINDOW_DAYS = 60  # only keep last 60 days of data

model_roll = xgb.train(XGB_PARAMS, dtrain, num_boost_round=1200, verbose_eval=False)

roll_maes = []
preds = model_roll.predict(dtest)
mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
roll_maes.append(mae_per_task)
print(f'After initial training | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f}')

seen_df = train_df.copy()

for i, batch_df in enumerate(batch_dfs):
    seen_df = pd.concat([seen_df, batch_df], ignore_index=True)
    cutoff_date = pd.to_datetime(seen_df['date'].max()) - pd.Timedelta(days=WINDOW_DAYS)
    window_df   = seen_df[pd.to_datetime(seen_df['date']) >= cutoff_date].copy()

    X_window = make_X_single(window_df, features, cat_cols, train_columns=train_columns)
    y_window = window_df['Time_Delta_sec'].astype(float)
    d_window = xgb.DMatrix(X_window, label=y_window)

    model_roll = xgb.train(
        XGB_PARAMS, d_window,
        num_boost_round=UPDATE_TREES,
        xgb_model=model_roll,
        verbose_eval=False,
    )
    preds = model_roll.predict(dtest)
    mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
    roll_maes.append(mae_per_task)
    n_dropped = len(seen_df) - len(window_df)
    print(f'After batch {i+1} | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f} | Rows in window: {len(window_df):,} | Dropped: {n_dropped:,}')

results['Rolling window (60d)'] = roll_maes
print('Done.')


In [ ]:
# ── Option 3a: KS-test drift detection ───────────────────────────────────
# Before each update, run a Kolmogorov-Smirnov test comparing the
# distribution of Time_Delta_sec in new data vs training data.
# If p < threshold: drift detected — down-weight pre-drift data.
# If p >= threshold: no drift — treat all data equally.
# Simple, explainable, no external dependencies.
print('Running KS-test drift detection strategy...')

KS_THRESHOLD   = 0.05   # p-value threshold — below this = drift detected
DRIFT_WEIGHT   = 0.3    # weight applied to pre-drift data when drift detected

model_ks = xgb.train(XGB_PARAMS, dtrain, num_boost_round=1200, verbose_eval=False)
reference_distribution = train_df['Time_Delta_sec'].dropna().values

ks_maes = []
preds = model_ks.predict(dtest)
mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
ks_maes.append(mae_per_task)
print(f'After initial training | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f}')

seen_df = train_df.copy()

for i, batch_df in enumerate(batch_dfs):
    # Run KS test: new batch distribution vs reference (training) distribution
    new_dist  = batch_df['Time_Delta_sec'].dropna().values
    ks_stat, p_value = stats.ks_2samp(reference_distribution, new_dist)
    drift_detected = p_value < KS_THRESHOLD

    print(f'  Batch {i+1} KS test: stat={ks_stat:.3f}, p={p_value:.4f} → Drift: {drift_detected}')

    seen_df = pd.concat([seen_df, batch_df], ignore_index=True)
    X_seen  = make_X_single(seen_df, features, cat_cols, train_columns=train_columns)
    y_seen  = seen_df['Time_Delta_sec'].astype(float)

    if drift_detected:
        # Down-weight rows from before the latest batch
        latest_batch_dates = set(batch_df['date'].unique())
        weights = np.where(
            seen_df['date'].isin(latest_batch_dates), 1.0, DRIFT_WEIGHT
        )
        print(f'    Drift detected — applying weight {DRIFT_WEIGHT} to pre-batch data')
    else:
        weights = np.ones(len(seen_df))
        print(f'    No drift — equal weights')

    d_seen = xgb.DMatrix(X_seen, label=y_seen, weight=weights)
    model_ks = xgb.train(
        XGB_PARAMS, d_seen,
        num_boost_round=UPDATE_TREES,
        xgb_model=model_ks,
        verbose_eval=False,
    )
    preds = model_ks.predict(dtest)
    mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
    ks_maes.append(mae_per_task)
    print(f'After batch {i+1} | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f}')

results['KS-test drift'] = ks_maes
print('Done.')


In [ ]:
# ── Option 3b: ADWIN drift detection ─────────────────────────────────────
# ADWIN (ADaptive WINdowing) is a streaming drift detector.
# It maintains a window of recent prediction errors and detects
# when the error distribution shifts significantly.
# More principled than KS-test — detects the exact change point.
#
# Install: pip install river
# river is the standard Python streaming ML library.
print('Running ADWIN drift detection strategy...')

try:
    from river import drift
    ADWIN_AVAILABLE = True
except ImportError:
    print('river not installed. Run: pip install river')
    print('Skipping ADWIN — install river and rerun this cell.')
    ADWIN_AVAILABLE = False

if ADWIN_AVAILABLE:
    ADWIN_DELTA    = 0.002  # sensitivity — lower = more sensitive to drift
    DRIFT_WEIGHT   = 0.3    # weight for pre-drift data when drift detected

    model_adwin = xgb.train(XGB_PARAMS, dtrain, num_boost_round=1200, verbose_eval=False)

    # Feed training errors through ADWIN to initialise the window
    adwin     = drift.ADWIN(delta=ADWIN_DELTA)
    X_train_d = xgb.DMatrix(X_train_init)
    train_preds = model_adwin.predict(X_train_d)
    train_errors = np.abs(train_df['Time_Delta_sec'].values - train_preds)
    for err in train_errors:
        adwin.update(err)

    adwin_maes = []
    preds = model_adwin.predict(dtest)
    mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
    adwin_maes.append(mae_per_task)
    print(f'After initial training | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f}')

    seen_df     = train_df.copy()
    drift_point = None

    for i, batch_df in enumerate(batch_dfs):
        # Get model predictions on new batch to compute errors
        X_batch  = make_X_single(batch_df, features, cat_cols, train_columns=train_columns)
        y_batch  = batch_df['Time_Delta_sec'].astype(float).values
        d_batch  = xgb.DMatrix(X_batch)
        b_preds  = model_adwin.predict(d_batch)
        b_errors = np.abs(y_batch - b_preds)

        # Feed errors into ADWIN — detects if error distribution shifted
        drift_detected = False
        for j, err in enumerate(b_errors):
            adwin.update(err)
            if adwin.drift_detected:
                drift_detected = True
                drift_point    = i  # batch where drift was detected
                print(f'  ADWIN drift detected in batch {i+1} at row {j}')
                break

        seen_df = pd.concat([seen_df, batch_df], ignore_index=True)
        X_seen  = make_X_single(seen_df, features, cat_cols, train_columns=train_columns)
        y_seen  = seen_df['Time_Delta_sec'].astype(float)

        if drift_detected:
            # Down-weight everything before the drift batch
            batch_dates = set(batch_df['date'].unique())
            weights = np.where(
                seen_df['date'].isin(batch_dates), 1.0, DRIFT_WEIGHT
            )
            print(f'    Applying weight {DRIFT_WEIGHT} to pre-drift data')
        else:
            weights = np.ones(len(seen_df))
            print(f'  Batch {i+1}: no drift detected — equal weights')

        d_seen = xgb.DMatrix(X_seen, label=y_seen, weight=weights)
        model_adwin = xgb.train(
            XGB_PARAMS, d_seen,
            num_boost_round=UPDATE_TREES,
            xgb_model=model_adwin,
            verbose_eval=False,
        )
        preds = model_adwin.predict(dtest)
        mae_per_task, r2, _ = eval_blocks(test_df, preds, train_columns)
        adwin_maes.append(mae_per_task)
        print(f'After batch {i+1} | MAE/task: {mae_per_task:.3f}s | R²: {r2:.4f}')

    results['ADWIN drift'] = adwin_maes
    print('Done.')


In [ ]:
# ── Comparison: MAE/task over time for all strategies ────────────────────
labels = ['Initial\ntraining', 'After\nbatch 1', 'After\nbatch 2', 'After\nbatch 3']

fig, ax = plt.subplots(figsize=(11, 6))

colors = {
    'Baseline':            'gray',
    'Naive update':        'steelblue',
    'Exp decay (λ=0.02)':  'darkorange',
    'Linear decay (90d)':  'forestgreen',
    'Rolling window (60d)':'crimson',
    'KS-test drift':       'purple',
    'ADWIN drift':         'brown',
}

for strategy, maes in results.items():
    x = range(len(maes))
    ax.plot(x, maes,
            marker='o', linewidth=2, markersize=7,
            label=strategy,
            color=colors.get(strategy, 'black'),
            linestyle='--' if strategy == 'Baseline' else '-')

ax.set_xticks(range(4))
ax.set_xticklabels(labels, fontsize=10)
ax.set_xlabel('Update stage', fontsize=12)
ax.set_ylabel('MAE per task (seconds)', fontsize=12)
ax.set_title(
    f'Incremental Learning Strategies — OE WC{WORKCODE}\n'
    f'Lower is better | Fixed test set | Block size = {BLOCK_SIZE} tasks',
    fontsize=13
)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary table
print(f'\nFinal MAE/task after all 3 batches:')
print(f'{"Strategy":<25} {"Initial":>10} {"Batch 1":>10} {"Batch 2":>10} {"Batch 3":>10}')
print('-' * 65)
for strategy, maes in results.items():
    vals = [f'{m:.3f}s' for m in maes]
    print(f'{strategy:<25} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10} {vals[3]:>10}')
